# Multi-Turn Conversational RAG Evaluation with RAGAS

**Problem:** Standard RAGAS metrics evaluate each question-answer pair independently. In production RAG systems, responses frequently reference information from prior conversation turns, causing false negatives when evaluated in isolation.

**Solution:** This notebook demonstrates `ConversationFaithfulness` and `ConversationRelevancy` metrics that evaluate RAG systems in conversational settings by considering both retrieved context AND prior conversation history.

**Learning Outcomes:**
- Understand why single-turn evaluation fails for conversational systems
- Implement multi-turn data models with sliding window history
- Design LLM prompts that leverage conversation context
- Build production-ready evaluation metrics
- Write comprehensive tests covering deterministic and integration cases

## Section 1: Environment Setup & Exploration

### 1.1 Install and Configure

```bash
# Install RAGAS with dev dependencies
make install-minimal  # Fast development setup

# Or manually:
uv sync --group dev

# Set up API keys
echo "OPENAI_API_KEY=sk-..." > .env
```

### 1.2 Understanding the Existing Architecture

The problem with standard metrics is **single-turn isolation**:

```python
# Standard Faithfulness evaluates this way:
Turn 1 context: "Elon Musk founded OpenAI in 2015"
Turn 2 context: "OpenAI had leadership changes in 2023"
Turn 2 question: "When did he leave?"
Turn 2 response: "Elon Musk left in 2018"

# Check: Is "Elon Musk" in Turn 2's context? NO → Score 0.0 ❌
# But: It's in Turn 1's history! Should be Score 1.0 ✓
```

Our solution adds **conversation-aware evaluation** that references prior turns.

In [ ]:
# First, let's set up our imports and verify the implementation
import sys
import asyncio
import json
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

# Verify we can import the new metrics
try:
    from ragas.metrics._conversation_metrics import (
        ConversationTurn,
        MultiTurnConversation,
        ConversationFaithfulness,
        ConversationRelevancy,
        ClaimVerdict,
        ClaimVerdicts,
        RelevancyVerdict,
    )
    print("✓ Successfully imported conversation metrics")
except ImportError as e:
    print(f"✗ Import failed: {e}")
    print("Make sure you have the latest version installed: pip install -e '.[dev]'")

# Check Python version
print(f"✓ Python version: {sys.version_info.major}.{sys.version_info.minor}")

## Section 2: Data Models for Multi-Turn Conversations

### 2.1 ConversationTurn Structure

Each turn in a conversation contains:
- **user_input**: The user's question/request
- **retrieved_contexts**: List of documents retrieved by RAG retriever
- **response**: The assistant's response  
- **turn_index**: Position in the conversation (0-indexed)

### 2.2 MultiTurnConversation & Sliding Window

The key innovation: we maintain a **configurable sliding window** over conversation history.

- **Default window_size=3**: Consider up to 3 prior turns as context
- **Motivation**: Balance accuracy vs token cost
  - Small window (1-2): Fast, cheaper, may miss important context
  - Large window (5+): More accurate, higher cost, may exceed context window
  - Default (3): Sweet spot for most conversations

In [ ]:
# Create a multi-turn conversation example
# This demonstrates the exact problem we're solving

conversation = MultiTurnConversation(
    conversation_id="openai-history-001",
    turns=[
        # Turn 1: Establish key facts
        ConversationTurn(
            user_input="Who founded OpenAI?",
            retrieved_contexts=[
                "OpenAI was founded in December 2015 by Sam Altman, Greg Brockman, "
                "Elon Musk, Ilya Sutskever, and others. It was established as a "
                "non-profit AI research laboratory with the goal of ensuring artificial "
                "general intelligence benefits all of humanity."
            ],
            response="OpenAI was founded by Sam Altman, Greg Brockman, Elon Musk, "
                     "and Ilya Sutskever in December 2015 as a non-profit research lab."
        ),
        
        # Turn 2: The problematic turn - references Turn 1 information
        ConversationTurn(
            user_input="When did Elon Musk leave the board?",
            # CRITICAL: This context does NOT mention Elon Musk!
            retrieved_contexts=[
                "OpenAI has experienced significant leadership changes. In 2019, "
                "OpenAI transitioned from a non-profit to a capped-profit structure. "
                "Microsoft has become a major strategic partner and investor in OpenAI."
            ],
            # But the response correctly references Elon Musk from Turn 1
            response="Elon Musk left the OpenAI board in 2018 citing potential "
                     "conflicts of interest with Tesla's autonomous vehicle work."
        ),
        
        # Turn 3: References information from Turn 1 & 2
        ConversationTurn(
            user_input="Who is the current CEO?",
            retrieved_contexts=[
                "Sam Altman has been serving as the CEO of OpenAI since 2019. "
                "Under his leadership, OpenAI developed GPT-4, a large language model "
                "that demonstrates advanced reasoning and problem-solving capabilities."
            ],
            response="Sam Altman, one of the original co-founders mentioned earlier, "
                     "has been leading OpenAI as CEO since 2019."
        ),
    ]
)

# Inspect the conversation structure
print(f"Conversation ID: {conversation.conversation_id}")
print(f"Total turns: {len(conversation.turns)}")
print()

# Demonstrate the sliding window mechanism
for turn_idx in range(len(conversation.turns)):
    turn = conversation.turns[turn_idx]
    history_turns = conversation.get_history_for_turn(turn_idx, window_size=3)
    history_text = conversation.format_history(turn_idx, window_size=3)
    
    print(f"{'='*70}")
    print(f"TURN {turn_idx + 1}")
    print(f"{'='*70}")
    print(f"Question: {turn.user_input}")
    print(f"\nHistory window size: {len(history_turns)} turns")
    if history_turns:
        print("History:")
        print(history_text)
    else:
        print("No prior conversation history")
    print()

## Section 3: LLM Prompt Design for Conversational Context

### 3.1 Three Core Prompts

The evaluation framework uses three specialized LLM prompts:

#### Prompt 1: Claim Extraction (Reuses RAGAS Pattern)
```
Given a response, extract all factual claims.
Return: {"claims": [...]}
```

#### Prompt 2: Multi-Turn Faithfulness Verification (NEW)
- Check if each claim is supported by **current context** OR **conversation history**
- Return per-claim verdicts with reasoning
- Critical: Must understand cross-turn references

#### Prompt 3: Conversation Relevancy (NEW)
- Evaluate if response addresses the question
- Check coherence with conversation history
- Detect contradictions with prior turns
- Return score 0.0-1.0 with reasoning

In [ ]:
# Step 1: Set up the LLM
# Using OpenAI as the LLM backend for evaluation
import os
from dotenv import load_dotenv

# Load API keys from .env
load_dotenv()

# Option 1: Using LangChain with OpenAI
try:
    from langchain_openai import ChatOpenAI
    from ragas.llms import LangchainLLMWrapper
    
    # Create OpenAI LLM instance
    openai_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    llm = LangchainLLMWrapper(openai_llm)
    print("✓ Successfully initialized OpenAI LLM via LangChain")
except ImportError:
    print("note: LangChain not available. Install with: pip install langchain-openai")

# Option 2: Using Anthropic Claude (alternative)
# from langchain_anthropic import ChatAnthropic
# from ragas.llms import LangchainLLMWrapper
# llm = LangchainLLMWrapper(ChatAnthropic(model="claude-3-5-sonnet-20241022"))

print(f"API Key loaded: {'OPENAI_API_KEY' in os.environ}")

In [ ]:
# Step 2: Initialize the metrics with custom window sizes
print("\\n" + "="*70)
print("INITIALIZING METRICS")
print("="*70)

# ConversationFaithfulness: Evaluates claims against context + history
faithfulness_metric = ConversationFaithfulness(
    window_size=3,  # Consider up to 3 prior turns
    llm=llm
)
print(f"✓ ConversationFaithfulness initialized (window_size={faithfulness_metric.window_size})")

# ConversationRelevancy: Evaluates response coherence with conversation
relevancy_metric = ConversationRelevancy(
    window_size=3,
    llm=llm
)
print(f"✓ ConversationRelevancy initialized (window_size={relevancy_metric.window_size})")

# You can also experiment with different window sizes
faithfulness_small_window = ConversationFaithfulness(window_size=1, llm=llm)
faithfulness_large_window = ConversationFaithfulness(window_size=5, llm=llm)
print(f"✓ Created variants with window_size=1 and window_size=5 for comparison")

In [ ]:
# Step 3: Run the evaluation - this is where the magic happens!

print("\\n" + "="*70)
print("EVALUATING CONVERSATION FAITHFULNESS")
print("="*70)
print("\\nThis will:")
print("1. Extract factual claims from each response")
print("2. Verify claims against CURRENT context and CONVERSATION HISTORY")
print("3. Compute per-turn scores and conversation average")
print("4. Show detailed claim-by-claim reasoning")
print()

async def run_evaluation():
    """Run the faithfulness evaluation with detailed results"""
    # Run with return_details=True to get per-claim reasoning
    result = await faithfulness_metric.score_conversation(
        conversation=conversation,
        return_details=True
    )
    return result

# Execute the async evaluation
result = asyncio.run(run_evaluation())

# Display results
print(f"Overall Conversation Faithfulness Score: {result['conversation_score']:.3f}")
print()

for turn_detail in result["turns"]:
    turn_idx = turn_detail["turn_index"]
    turn_score = turn_detail["score"]
    
    print(f"{'─'*70}")
    print(f"TURN {turn_idx + 1} | Score: {turn_score:.3f}")
    print(f"{'─'*70}")
    print(f"Question: {conversation.turns[turn_idx].user_input}")
    print(f"Response: {conversation.turns[turn_idx].response[:100]}...")
    print()
    
    # Show claim-by-claim analysis
    verdicts = turn_detail["verdicts"]
    if verdicts:
        print(f"Claims verified ({len(verdicts)} total):")
        for i, v in enumerate(verdicts, 1):
            status = "✓ FAITHFUL" if v["verdict"] == 1 else "✗ NOT FAITHFUL"
            print(f"  {i}. {v['claim'][:60]}...")
            
            sources = []
            if v["supported_by_context"]:
                sources.append("context")
            if v["supported_by_history"]:
                sources.append("history")
            source_str = " + ".join(sources) if sources else "NONE"
            
            print(f"     → Supported by: {source_str}")
            print(f"     → {status} ({v['reason']})")
    else:
        print("No claims extracted from response (may be too generic)")
    print()

In [ ]:
# Step 4: Run ConversationRelevancy evaluation
print("\\n" + "="*70)
print("EVALUATING CONVERSATION RELEVANCY")
print("="*70)
print("\\nThis evaluates whether responses:")
print("- Address the current question")
print("- Are coherent with conversation history")
print("- Don't contradict prior turns")
print()

async def run_relevancy_evaluation():
    """Run the relevancy evaluation"""
    result = await relevancy_metric.score_conversation(
        conversation=conversation,
        return_details=True
    )
    return result

relevancy_result = asyncio.run(run_relevancy_evaluation())

print(f"Overall Conversation Relevancy Score: {relevancy_result['conversation_score']:.3f}")
print()

for turn_detail in relevancy_result["turns"]:
    turn_idx = turn_detail["turn_index"]
    score = turn_detail["score"]
    
    print(f"{'─'*70}")
    print(f"TURN {turn_idx + 1} | Relevancy Score: {score:.3f}")
    print(f"{'─'*70}")
    print(f"Question: {conversation.turns[turn_idx].user_input}")
    print(f"Addresses question: {'✓ Yes' if turn_detail['addresses_question'] else '✗ No'}")
    print(f"Coherent with history: {'✓ Yes' if turn_detail['coherent_with_history'] else '✗ No'}")
    print(f"Reasoning: {turn_detail['reason']}")
    print()

## Section 7: Unit Testing - Deterministic Cases

Unit tests verify metric behavior WITHOUT requiring LLM calls. This ensures fast, reliable testing.

In [ ]:
# Unit Test 1: Data Model Tests (No LLM needed)
print("="*70)
print("UNIT TEST 1: Conversation Data Models")
print("="*70)

# Test 1.1: ConversationTurn formatting
turn = ConversationTurn(
    user_input="What is Python?",
    retrieved_contexts=["Python is a programming language"],
    response="Python is a programming language.",
    turn_index=2
)
formatted = turn.format_for_history()
assert "Turn 3" in formatted  # turn_index=2 means Turn 3
assert "What is Python?" in formatted
assert "Python is a programming language." in formatted
print("✓ Test 1.1: ConversationTurn.format_for_history() works correctly")

# Test 1.2: MultiTurnConversation assigns turn indices
test_conv = MultiTurnConversation(
    turns=[
        ConversationTurn(user_input="q1", retrieved_contexts=[], response="r1"),
        ConversationTurn(user_input="q2", retrieved_contexts=[], response="r2"),
        ConversationTurn(user_input="q3", retrieved_contexts=[], response="r3"),
    ]
)
for i, turn in enumerate(test_conv.turns):
    assert turn.turn_index == i
print("✓ Test 1.2: Turn indices assigned correctly (0, 1, 2, ...)")

# Test 1.3: History window size limits
history_turn_1 = test_conv.get_history_for_turn(1)
assert len(history_turn_1) == 1  # Turn 1 has 1 prior turn
assert history_turn_1[0].turn_index == 0

history_turn_2_window1 = test_conv.get_history_for_turn(2, window_size=1)
assert len(history_turn_2_window1) == 1  # Only 1 turn in window
assert history_turn_2_window1[0].turn_index == 1  # Most recent

history_turn_2_window5 = test_conv.get_history_for_turn(2, window_size=5)
assert len(history_turn_2_window5) == 2  # Both prior turns (not 5)
assert history_turn_2_window5[0].turn_index == 0
assert history_turn_2_window5[1].turn_index == 1
print("✓ Test 1.3: Sliding window correctly limits history")

# Test 1.4: No history for first turn
history_turn_0 = test_conv.get_history_for_turn(0)
assert len(history_turn_0) == 0
print("✓ Test 1.4: First turn has no history")

In [ ]:
# Unit Test 2: Metric Score Computation (No LLM needed)
print("\\n" + "="*70)
print("UNIT TEST 2: Metric Score Computation")
print("="*70)

metric = ConversationFaithfulness(llm=llm)

# Test 2.1: All claims faithful = 1.0
verdicts_all_faithful = ClaimVerdicts(verdicts=[
    ClaimVerdict(claim="c1", supported_by_context=True, supported_by_history=False, 
                 verdict=1, reason="Found in context"),
    ClaimVerdict(claim="c2", supported_by_context=False, supported_by_history=True,
                 verdict=1, reason="Found in history"),
])
score = metric._compute_turn_score(verdicts_all_faithful)
assert score == 1.0, f"Expected 1.0, got {score}"
print("✓ Test 2.1: All faithful claims → score = 1.0")

# Test 2.2: No claims faithful = 0.0
verdicts_no_faithful = ClaimVerdicts(verdicts=[
    ClaimVerdict(claim="c1", supported_by_context=False, supported_by_history=False,
                 verdict=0, reason="Not supported"),
    ClaimVerdict(claim="c2", supported_by_context=False, supported_by_history=False,
                 verdict=0, reason="Not supported"),
])
score = metric._compute_turn_score(verdicts_no_faithful)
assert score == 0.0, f"Expected 0.0, got {score}"
print("✓ Test 2.2: No faithful claims → score = 0.0")

# Test 2.3: Partial faithfulness = 0.5
verdicts_partial = ClaimVerdicts(verdicts=[
    ClaimVerdict(claim="c1", supported_by_context=True, supported_by_history=False,
                 verdict=1, reason="Ok"),
    ClaimVerdict(claim="c2", supported_by_context=False, supported_by_history=False,
                 verdict=0, reason="Not supported"),
])
score = metric._compute_turn_score(verdicts_partial)
assert score == 0.5, f"Expected 0.5, got {score}"
print("✓ Test 2.3: 50% faithful claims → score = 0.5")

# Test 2.4: No claims = 1.0 (vacuously true)
verdicts_empty = ClaimVerdicts(verdicts=[])
score = metric._compute_turn_score(verdicts_empty)
assert score == 1.0, f"Expected 1.0 for empty, got {score}"
print("✓ Test 2.4: No claims extracted → score = 1.0 (vacuously true)")

# Test 2.5: Various claim counts
for n_claims, n_faithful in [(10, 10), (10, 8), (10, 5), (10, 1), (10, 0)]:
    verdicts = ClaimVerdicts(verdicts=[
        ClaimVerdict(
            claim=f"c{i}",
            supported_by_context=i < n_faithful,
            supported_by_history=False,
            verdict=1 if i < n_faithful else 0,
            reason="test"
        ) for i in range(n_claims)
    ])
    score = metric._compute_turn_score(verdicts)
    expected = n_faithful / n_claims
    assert abs(score - expected) < 1e-6, f"Expected {expected}, got {score}"
print(f"✓ Test 2.5: Score computation correct for various claim counts")

print("\\n✓ All metric computation tests passed!")

## Section 8: Impact of Sliding Window Size

The window_size parameter has significant impact on metric behavior. Let's explore this trade-off.

In [ ]:
# Analyze the relationship between window size and token cost vs accuracy
import numpy as np

print("="*70)
print("WINDOW SIZE ANALYSIS")
print("="*70)
print()
print("Window Size Trade-offs:")
print("  Size=1: Fast (cheapest), but may miss important context")
print("  Size=3: Balanced (default), good for most conversations")
print("  Size=5: Thorough, higher token cost, may exceed context window")
print()

# Show history sizes for each window configuration
for window_size in [1, 3, 5]:
    print(f"\\nWindow Size = {window_size}:")
    total_history_tokens = 0
    for turn_idx in range(len(conversation.turns)):
        history = conversation.get_history_for_turn(turn_idx, window_size)
        history_text = conversation.format_history(turn_idx, window_size)
        # Rough token estimate: ~4 characters per token
        history_tokens = len(history_text) // 4
        total_history_tokens += history_tokens
        print(f"  Turn {turn_idx + 1}: {len(history)} prior turns, " +
              f"~{history_tokens} tokens of history")
    
    print(f"  Total estimated tokens (all turns): {total_history_tokens}")
    print(f"  Cost per evaluation: ~${(total_history_tokens * 0.000001):.6f} " +
          f"(GPT-4o mini rates)")

print("\\n" + "="*70)
print("RECOMMENDATION")
print("="*70)
print("""
For most applications:
  ✓ Use window_size=3 (default) as starting point
  ✓ Monitor token costs and accuracy on your specific domain
  ✓ Adjust based on:
    - Conversation length (longer → larger window needed)
    - Context window size of your LLM (transformer limit)
    - Cost constraints and latency requirements
""")

## Summary: What We Built

### Key Achievements

1. **ConversationFaithfulness Metric**
   - Evaluates claims against both current context AND conversation history
   - Solves false negatives when answers reference prior turns
   - Uses LLM-as-judge pattern consistent with RAGAS

2. **ConversationRelevancy Metric**
   - Validates that responses address the current question
   - Checks coherence with established conversation context
   - Detects contradictions with prior turns

3. **Data Models**
   - `ConversationTurn`: Represents single turn with context
   - `MultiTurnConversation`: Manages sliding window history
   - Configurable window size (default=3) balances accuracy vs cost

4. **Comprehensive Testing**
   - Unit tests verify data models (zero LLM dependency)
   - Score computation tests with mock verdicts
   - Integration tests with real LLM (optional)

### Real-World Impact

**Before (Standard Metrics):**
```
Turn 2: "When did Elon Musk leave?"
Response: "Elon Musk left in 2018"
Turn 2 Context: [Leadership changes in 2023...]
Score: 0.0 ❌ (Name not in current context)
```

**After (Conversation-Aware Metrics):**
```
Turn 2: "When did Elon Musk leave?"
Response: "Elon Musk left in 2018"
Check: History has "Elon Musk founded OpenAI" ✓
Score: 1.0 ✓ (Correctly recognizes cross-turn reference)
```